## Overview

Spline-based selectivity provides a flexible, semi-parametric
alternative to logistic functional forms. Rather than imposing a
specific shape (monotone or dome-shaped), a B-spline basis is used to
represent selectivity as a smooth function of age, with the degree of
smoothness controlled by a penalty on the second differences of the
spline coefficients.

This formulation is consistent with a broader selectivity-estimation
literature that emphasizes flexible shape representation together with
explicit regularization (Millar and Fryer 1999; Wileman et al. 1996). In
assessment applications, spline penalties act as the key control on
effective complexity and therefore on bias-variance trade-offs (Martell
and Stewart 2014).

## Model definition

Let $\mathbf{B}(a)$ be a B-spline basis matrix evaluated at ages $a$,
with $K$ basis functions and coefficient vector $\boldsymbol{\beta}$.
The log-selectivity is

$$
\log\,\text{sel}(a) = \mathbf{B}(a)\,\boldsymbol{\beta}
$$

with selectivity normalized so the mean over a reference age range
equals 1:

$$
\text{sel}(a) = \frac{\exp(\mathbf{B}(a)\,\boldsymbol{\beta})}
{\frac{1}{|\mathcal{A}_{\text{ref}}|}\sum_{a' \in \mathcal{A}_{\text{ref}}}
\exp(\mathbf{B}(a')\,\boldsymbol{\beta})}.
$$

Here, $\mathcal{A}_{\text{ref}}$ is a user-specified reference age range
(typically ages expected to be most highly selected). This normalization
keeps the objective smooth and differentiable, avoiding the
non-differentiability introduced by a $\max(\cdot)$ denominator.

Smoothness is enforced via a penalty on second differences:

$$
\text{penalty} = \frac{\lambda}{2} \sum_{k=3}^{K} (\beta_k - 2\beta_{k-1} + \beta_{k-2})^2
$$

where $\lambda$ controls the trade-off between fit and smoothness.

In [ ]:
library(ggplot2)
library(dplyr)
library(splines)

ages <- seq(1, 15, by = 0.1)

## Basis visualization

In [ ]:
K <- 6
B <- bs(ages, df = K, degree = 3, intercept = TRUE)
basis_df <- as.data.frame(B) %>%
  mutate(age = ages) %>%
  tidyr::pivot_longer(-age, names_to = "basis", values_to = "value")

ggplot(basis_df, aes(x = age, y = value, color = basis)) +
  geom_line(linewidth = 1) +
  labs(x = "Age", y = "Basis function value",
       title = "B-spline basis (degree 3, K = 6)") +
  theme_bw(base_size = 11) +
  theme(legend.position = "none", panel.grid.minor = element_blank())

## Example curves

In [ ]:
ref_age_min <- 8
ref_age_max <- 12

spline_sel <- function(age, beta, B) {
  log_sel <- as.numeric(B %*% beta)
  sel_raw <- exp(log_sel)
  ref_idx <- age >= ref_age_min & age <= ref_age_max
  sel <- sel_raw / mean(sel_raw[ref_idx])
  as.numeric(sel)
}

B_mat <- bs(ages, df = K, degree = 3, intercept = TRUE)

examples <- tibble(
  name = c("Logistic-like", "Dome-shaped", "Bimodal", "Flat-topped"),
  beta = list(
    c(-3, -1, 0, 1, 1, 1),
    c(-3, -1, 0, 1, 0, -1),
    c(-1, 1, -0.5, 0, 1, -0.5),
    c(-3, -1, 0, 0, 0, 0)
  )
)

curve_data <- examples %>%
  rowwise() %>%
  do({
    ex <- .
    tibble(
      name = ex$name,
      age = ages,
      selectivity = spline_sel(ages, ex$beta, B_mat)
    )
  }) %>%
  ungroup()

ggplot(curve_data, aes(x = age, y = selectivity, color = name)) +
  geom_line(linewidth = 1.2) +
  scale_y_continuous(limits = c(0, NA)) +
  scale_color_brewer(palette = "Set2", name = "Shape") +
  labs(x = "Age", y = "Relative selectivity (reference-age mean = 1)",
       title = "Spline selectivity: example shapes") +
  theme_bw(base_size = 11) +
  theme(panel.grid.minor = element_blank())

## RTMB implementation

*Placeholder: RTMB objective with spline coefficients as parameters,
smoothness penalty as a tuning parameter or estimated, and MCMC
sampling.*

In [ ]:
library(RTMB)

sel_ref_min <- 8
sel_ref_max <- 12

# Objective function skeleton
f <- function(parms) {
  getAll(data, parms, warn = FALSE)
  log_sel <- B %*% beta
  sel_raw <- exp(log_sel)
  ref_idx <- age >= sel_ref_min & age <= sel_ref_max
  sel_hat <- sel_raw / mean(sel_raw[ref_idx])

  # Second-difference penalty
  K <- length(beta)
  d2 <- beta[3:K] - 2 * beta[2:(K-1)] + beta[1:(K-2)]
  penalty <- 0.5 * exp(log_lambda) * sum(d2^2)

  nll <- penalty
  # Add data likelihood here
  # nll <- nll - sum(dnorm(sel_obs, sel_hat, sel_sd, log = TRUE))

  ADREPORT(sel_hat)
  nll
}

Martell, Steven J. D., and Ian J. Stewart. 2014. “Towards Defining Good
Practices for Modeling Time-Varying Selectivity.” *Fisheries Research*
158: 84–95. <https://doi.org/10.1016/j.fishres.2013.11.001>.

Millar, R. B., and R. J. Fryer. 1999. “Estimating the Size-Selection
Curves of Towed Gears, Traps, Nets and Hooks.” *Reviews in Fish Biology
and Fisheries* 9: 89–116. <https://doi.org/10.1023/A:1008838220001>.

Wileman, D. A., R. S. T. Ferro, R. Fonteyne, and R. B. Millar. 1996.
“Manual of Methods of Measuring the Selectivity of Towed Fishing Gears.”
ICES Cooperative Research Report 215. Copenhagen: International Council
for the Exploration of the Sea (ICES).
<https://ices-library.figshare.com/articles/report/Manual_of_methods_of_measuring_the_selectivity_of_towed_fishing_gears/18624446/1/files/33403508.pdf>.